# 02 · Feature engineering

Build rolling features per base signal (mean, std, min, max, slope, EMA over windows 5, 10, 20): 342 features.

- **Reads:** notebook 01 splits
- **Writes:** `<dataset>_<split>_features.parquet`, `feature_columns.json`, `feature_manifest.csv`

Written for Kaggle (`/kaggle/input/...`, `/kaggle/working/...`; the Kaggle account name is replaced by `<kaggle-user>`). Outputs are kept as saved. Read `docs/PIPELINE.md` for the order and for what is missing.

In [1]:
# ==========================================================
# NB02 : FEATURE ENGINEERING
# ==========================================================

import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

WORKING_DIR = Path("/kaggle/working")

NB01_DIR = Path(
    "/kaggle/input/datasets/<kaggle-user>/nb-1-again-outputs"
)

OUTPUT_ROOT = WORKING_DIR / "NB-2-AGAIN-outputs"

FEATURE_DIR = OUTPUT_ROOT / "features"
METADATA_DIR = OUTPUT_ROOT / "metadata"
LOG_DIR = OUTPUT_ROOT / "logs"

for d in [FEATURE_DIR, METADATA_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATASETS = ["FD001", "FD002", "FD003", "FD004"]
SPLITS = ["train", "val_a", "val_b", "test"]

ROLLING_WINDOWS = [5, 10, 20]

print("✅ CELL 1 DONE")

✅ CELL 1 DONE


In [2]:
sensor_metadata_path = (
    NB01_DIR / "metadata" / "/kaggle/input/datasets/<kaggle-user>/nb-1-again-outputs/sensor_metadata.json"
)

with open(sensor_metadata_path, "r") as f:
    sensor_metadata = json.load(f)

ACTIVE_SENSOR_COLS = sensor_metadata["active_sensors"]
SENSORS_REMOVED = sensor_metadata["sensors_removed"]
OPERATING_SETTINGS = sensor_metadata["operating_settings"]

assert len(ACTIVE_SENSOR_COLS) == 15, (
    f"Expected 15 active sensors, got {len(ACTIVE_SENSOR_COLS)}"
)

expected_removed = {
    "sensor_1",
    "sensor_5",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19"
}

assert set(SENSORS_REMOVED) == expected_removed

BASE_FEATURE_COLS = (
    OPERATING_SETTINGS + ACTIVE_SENSOR_COLS
)

assert len(BASE_FEATURE_COLS) == 18

print("Active sensors:", len(ACTIVE_SENSOR_COLS))
print("Base features:", len(BASE_FEATURE_COLS))

print("✅ Sensor configuration validated")
print("✅ CELL 2 DONE")

Active sensors: 15
Base features: 18
✅ Sensor configuration validated
✅ CELL 2 DONE


In [3]:
INDEX_COLS = [
    "engine_id",
    "cycle"
]

LABEL_COLS = [
    "rul",
    "failure_within_horizon"
]

ENGINEERED_FEATURES = [
    "mean",
    "std",
    "min",
    "max",
    "slope",
    "ema"
]

EXPECTED_FEATURE_COUNT = 342

print("✅ CELL 3 DONE")

✅ CELL 3 DONE


In [4]:
def calculate_slope(values):

    if len(values) < 2:
        return 0.0

    x = np.arange(len(values))

    return np.polyfit(x, values, 1)[0]


def create_features(df):

    df = df.sort_values(
        ["engine_id", "cycle"]
    ).copy()

    grouped = df.groupby("engine_id")

    feature_frames = []

    for col in BASE_FEATURE_COLS:

        feature_frames.append(
            grouped[col]
            .transform(lambda x: x)
            .rename(col)
        )

        for window in ROLLING_WINDOWS:

            roll = grouped[col].rolling(
                window=window,
                min_periods=1
            )

            feature_frames.append(
                roll.mean()
                .reset_index(level=0, drop=True)
                .rename(f"{col}_mean_w{window}")
            )

            feature_frames.append(
                roll.std()
                .reset_index(level=0, drop=True)
                .fillna(0)
                .rename(f"{col}_std_w{window}")
            )

            feature_frames.append(
                roll.min()
                .reset_index(level=0, drop=True)
                .rename(f"{col}_min_w{window}")
            )

            feature_frames.append(
                roll.max()
                .reset_index(level=0, drop=True)
                .rename(f"{col}_max_w{window}")
            )

            feature_frames.append(
                roll.apply(
                    calculate_slope,
                    raw=True
                )
                .reset_index(level=0, drop=True)
                .fillna(0)
                .rename(f"{col}_slope_w{window}")
            )

            feature_frames.append(
                grouped[col]
                .transform(
                    lambda x: x.ewm(
                        span=window,
                        adjust=False
                    ).mean()
                )
                .rename(f"{col}_ema_w{window}")
            )

    engineered_df = pd.concat(
        feature_frames,
        axis=1
    )

    final_df = pd.concat(
        [
            df[INDEX_COLS + LABEL_COLS].reset_index(drop=True),
            engineered_df.reset_index(drop=True)
        ],
        axis=1
    )

    return final_df

In [5]:
manifest = []
all_feature_columns = None

for dataset in DATASETS:

    for split in SPLITS:

        print(f"\nProcessing {dataset} | {split}")

        input_path = (
            NB01_DIR
            / f"{dataset}_{split}.parquet"
        )

        df = pd.read_parquet(input_path)

        feature_df = create_features(df)

        feature_cols = [
            c for c in feature_df.columns
            if c not in INDEX_COLS + LABEL_COLS
        ]

        actual_feature_count = len(feature_cols)

        assert actual_feature_count == EXPECTED_FEATURE_COUNT, (
            f"{dataset}-{split}: "
            f"Expected {EXPECTED_FEATURE_COUNT}, "
            f"got {actual_feature_count}"
        )

        output_path = (
            FEATURE_DIR
            / f"{dataset}_{split}_features.parquet"
        )

        feature_df.to_parquet(
            output_path,
            index=False
        )

        manifest.append({
            "dataset": dataset,
            "split": split,
            "rows": len(feature_df),
            "features": actual_feature_count
        })

        if all_feature_columns is None:
            all_feature_columns = feature_cols

        print(
            f"Rows: {len(feature_df):,}"
        )

        print(
            f"Features: {actual_feature_count}"
        )

print("\n✅ CELL 5 DONE")


Processing FD001 | train
Rows: 14,507
Features: 342

Processing FD001 | val_a
Rows: 1,759
Features: 342

Processing FD001 | val_b
Rows: 1,007
Features: 342

Processing FD001 | test
Rows: 3,358
Features: 342

Processing FD002 | train
Rows: 38,255
Features: 342

Processing FD002 | val_a
Rows: 5,002
Features: 342

Processing FD002 | val_b
Rows: 2,510
Features: 342

Processing FD002 | test
Rows: 7,992
Features: 342

Processing FD003 | train
Rows: 17,187
Features: 342

Processing FD003 | val_a
Rows: 2,007
Features: 342

Processing FD003 | val_b
Rows: 1,245
Features: 342

Processing FD003 | test
Rows: 4,281
Features: 342

Processing FD004 | train
Rows: 42,834
Features: 342

Processing FD004 | val_a
Rows: 5,974
Features: 342

Processing FD004 | val_b
Rows: 3,066
Features: 342

Processing FD004 | test
Rows: 9,375
Features: 342

✅ CELL 5 DONE


In [6]:
manifest_df = pd.DataFrame(manifest)

manifest_df.to_csv(
    METADATA_DIR / "feature_manifest.csv",
    index=False
)

with open(
    METADATA_DIR / "feature_columns.json",
    "w"
) as f:
    json.dump(all_feature_columns, f, indent=4)

feature_metadata = {
    "rolling_windows": ROLLING_WINDOWS,
    "engineered_features": ENGINEERED_FEATURES,
    "base_features": len(BASE_FEATURE_COLS),
    "active_sensors": len(ACTIVE_SENSOR_COLS),
    "total_features": EXPECTED_FEATURE_COUNT
}

with open(
    METADATA_DIR / "feature_metadata.json",
    "w"
) as f:
    json.dump(feature_metadata, f, indent=4)

print("✅ CELL 6 DONE")

✅ CELL 6 DONE


In [7]:
feature_files = list(
    FEATURE_DIR.glob("*.parquet")
)

assert len(feature_files) == 16

for file in feature_files:

    df = pd.read_parquet(file)

    feature_cols = [
        c for c in df.columns
        if c not in INDEX_COLS + LABEL_COLS
    ]

    assert len(feature_cols) == 342

    assert not df.isnull().any().any()

print(f"Validated files: {len(feature_files)}")

print("✅ CELL 7 DONE")

Validated files: 16
✅ CELL 7 DONE


In [8]:
summary = f"""
NB02 FEATURE ENGINEERING COMPLETE

Active Sensors: {len(ACTIVE_SENSOR_COLS)}
Base Features: {len(BASE_FEATURE_COLS)}

Rolling Windows: {ROLLING_WINDOWS}

Features per variable: 19
Variables: {len(BASE_FEATURE_COLS)}

Total Features: {EXPECTED_FEATURE_COUNT}
Files Generated: 16
"""

with open(
    LOG_DIR / "nb02_summary.txt",
    "w"
) as f:
    f.write(summary)

print(summary)

print("✅ CELL 8 DONE")


NB02 FEATURE ENGINEERING COMPLETE

Active Sensors: 15
Base Features: 18

Rolling Windows: [5, 10, 20]

Features per variable: 19
Variables: 18

Total Features: 342
Files Generated: 16

✅ CELL 8 DONE


In [9]:
zip_path = shutil.make_archive(
    str(OUTPUT_ROOT),
    "zip",
    OUTPUT_ROOT
)

print(f"ZIP CREATED: {zip_path}")

print("🎉 NB02 COMPLETED SUCCESSFULLY")
print("✅ CELL 9 DONE") 

ZIP CREATED: /kaggle/working/NB-2-AGAIN-outputs.zip
🎉 NB02 COMPLETED SUCCESSFULLY
✅ CELL 9 DONE
